# Engramma Memory — Building a Chatbot with Long-Term Memory

This notebook shows how to give a chatbot **persistent, compositional memory**.

The bot remembers past conversations and can **blend related memories** to build richer context — something vector databases cannot do natively.

In [ ]:
import numpy as np
from engramma_memory import EngrammaMemory

## Embedding Function

In production, replace this with your model (OpenAI `text-embedding-3-small`, sentence-transformers, etc.).
This deterministic hash-based embedding lets us demonstrate without API keys.

In [ ]:
DIM = 128

def embed(text: str) -> np.ndarray:
    """Deterministic pseudo-embedding for demo purposes."""
    rng = np.random.default_rng(hash(text) % (2**32))
    vec = rng.standard_normal(DIM).astype(np.float32)
    return vec / (np.linalg.norm(vec) + 1e-8)

## The ChatbotWithMemory Class

In [ ]:
class ChatbotWithMemory:
    def __init__(self):
        self.mem = EngrammaMemory(dim=DIM, backend="local")
        self.history: list = []

    def remember(self, text: str):
        """Store a message in long-term memory."""
        embedding = embed(text)
        self.mem.store(key=embedding, value=embedding)
        self.history.append(text)

    def recall(self, query: str, top_k: int = 3) -> list:
        """Find relevant past messages."""
        if self.mem.count == 0:
            return []
        embedding = embed(query)
        results = self.mem.query(embedding, top_k=top_k)
        return [self._match(r["value"]) for r in results if r["score"] > 0.1]

    def compose_context(self, topics: list) -> str:
        """Blend multiple topics into unified context (Engramma-exclusive)."""
        embeddings = [embed(t) for t in topics]
        composed = self.mem.compose(embeddings)
        return self._match(composed)

    def _match(self, value: np.ndarray) -> str:
        """Find closest stored message to a value vector."""
        value = value.flatten()[:DIM]
        best_dist, best_text = float('inf'), ""
        for text in self.history:
            dist = float(np.linalg.norm(embed(text) - value))
            if dist < best_dist:
                best_dist, best_text = dist, text
        return best_text

## Simulate a Conversation

In [ ]:
bot = ChatbotWithMemory()

# User tells the bot various things over multiple sessions
messages = [
    "I love programming in Python",
    "Machine learning is fascinating",
    "I'm building a recommendation system",
    "The weather is nice today",
    "I need help with PyTorch",
    "My project deadline is next Friday",
    "I prefer VS Code over PyCharm",
    "Data preprocessing takes most of my time",
]

print("=== Storing messages ===")
for msg in messages:
    bot.remember(msg)
    print(f"  ✓ '{msg}'")

print(f"\nTotal memories: {bot.mem.count}")

## Simple Recall

Find memories relevant to a query — similar to what a vector DB does.

In [ ]:
print("Query: 'What do I know about AI projects?'")
recalled = bot.recall("What do I know about AI projects?", top_k=3)
for r in recalled:
    print(f"  → {r}")

print("\nQuery: 'coding tools'")
recalled = bot.recall("coding tools", top_k=2)
for r in recalled:
    print(f"  → {r}")

## Compositional Context (Engramma Advantage)

This is the key differentiator. Instead of retrieving separately for "Python" and "machine learning" and concatenating results, Engramma **composes** them through multi-head attention.

The result is the memory that best **bridges** both topics simultaneously.

In [ ]:
print("Compose: ['Python', 'machine learning']")
context = bot.compose_context(["Python", "machine learning"])
print(f"  → {context}")

print("\nCompose: ['deadline', 'project']")
context = bot.compose_context(["deadline", "project"])
print(f"  → {context}")

print("\nCompose: ['Python', 'tools', 'productivity']")
context = bot.compose_context(["Python", "tools", "productivity"])
print(f"  → {context}")

## Forgetting Old Information

When user preferences change, decay or remove old memories.

In [ ]:
# User updates: "Actually, I switched to JAX"
bot.remember("I switched from PyTorch to JAX")

# Decay the old PyTorch memory
old_embedding = embed("I need help with PyTorch")
bot.mem.forget(old_embedding, strategy="decay")

print("After update:")
recalled = bot.recall("deep learning framework", top_k=2)
for r in recalled:
    print(f"  → {r}")

## Integration with an LLM

In production, you'd feed the recalled/composed context into your LLM prompt:

In [ ]:
def build_prompt(user_message: str, bot: ChatbotWithMemory) -> str:
    """Build an LLM prompt enriched with Engramma memory."""
    # Recall relevant memories
    memories = bot.recall(user_message, top_k=5)
    
    prompt = "You are a helpful assistant with memory of past conversations.\n\n"
    if memories:
        prompt += "Relevant memories:\n"
        for m in memories:
            prompt += f"- {m}\n"
        prompt += "\n"
    prompt += f"User: {user_message}\nAssistant:"
    return prompt

# Example
prompt = build_prompt("What project should I work on next?", bot)
print(prompt)

## Memory Stats

In [ ]:
stats = bot.mem.stats()
print("Memory Statistics:")
for k, v in stats.items():
    print(f"  {k}: {v}")

## Production: Switch to Cloud

For unlimited memory, persistence, and weighted composition:

```python
class ProductionBot(ChatbotWithMemory):
    def __init__(self):
        self.mem = EngrammaMemory(dim=DIM, backend="cloud", api_key="nx_live_...")
        self.history = []
```

Everything else stays the same. Get your key at [www.engramma-memory.com/signup](https://www.engramma-memory.com/signup).